# Lenguajes y Autómatas II (SCD-1016)
## Sesión 34 · Marco de Pila (Stack Frame) y Prólogo/Epílogo

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se construye y se destruye el espacio de trabajo de una función en la pila?
>
> **Objetivo de la sesión:** Calcular el tamaño alineado del marco para que la pila quede lista para llamar a libc y reconocer la secuencia con la que una función se despide.

**Entregable:** tu implementación de marco_alineado() y epilogo() en artesano_sesion34_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion34.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes cuántos bytes de variables locales y cuántos registros callee-saved hay que preservar; devuelves la N exacta de subq $N, %rsp. También devuelves la secuencia de salida de la función.

Al final de la sesión tendrás: **Una función que calcula el tamaño de marco alineado a 16 bytes y otra que devuelve el epílogo compacto, ambas auditadas.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 34](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion34.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion34.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion34.h"

int marco_alineado(int locales, int callee_saved);
const char *epilogo(void);
```

- El prólogo ya empujó %rbp y los k registros callee-saved antes de restar N a %rsp.
- N debe ser el menor valor que cubra las locales y deje %rsp alineado a 16 bytes.
- Un número negativo de locales o de registros se rechaza con -1.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion34_c.c` junto con `artesano_sesion34_c.c`
> (o `ia_sesion34_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `marco_alineado(0, 0)` | 0 |
| 2 | `marco_alineado(8, 0)` | 16 |
| 3 | `marco_alineado(24, 0)` | 32 |
| 4 | `marco_alineado(8, 1)` | 8 |
| 5 | `marco_alineado(-1, 0)` | -1 |
| 6 | `epilogo()` | `leave; ret` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 34 - Configuracion de punteros RBP y RSP ===
Caso                               esperado:              obtenido:              resultado
[ 1] marco_alineado(0, 0)               esperado: 0            obtenido: 0            OK
[ 2] marco_alineado(8, 0)               esperado: 16           obtenido: 16           OK
[ 3] marco_alineado(24, 0)              esperado: 32           obtenido: 32           OK
[ 4] marco_alineado(8, 1)               esperado: 8            obtenido: 8            OK
[ 5] marco_alineado(-1, 0)              esperado: -1           obtenido: -1           OK
[ 6] epilogo()                          esperado: leave; ret   obtenido: leave; ret   OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion34.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] marco_alineado devuelve la N mínima que cubre las locales y deja %rsp en múltiplo de 16.
- [ ] epilogo devuelve exactamente la secuencia compacta leave; ret, sin texto adicional.
- [ ] Los seis casos del harness salen OK y el programa termina con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion34_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion34.h
/* api_sesion34.h - contrato de la Sesion 34
 * Configuración de punteros RBP y RSP
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION34_H
#define API_SESION34_H

int marco_alineado(int locales, int callee_saved);
const char *epilogo(void);

#endif /* API_SESION34_H */


In [ ]:
%%writefile artesano_sesion34_c.c
/* artesano_sesion34_c.c - Fase 1: El Artesano (sin IA)
 * Configuración de punteros RBP y RSP.
 *
 * Implementa lo que declara api_sesion34.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion34.h"
#include <stdio.h>
#include <string.h>

int marco_alineado(int locales, int callee_saved) {
    (void)locales;
    (void)callee_saved; /* TODO: redondear segun la alineacion de la ABI */
    return 0;
}

const char *epilogo(void) {
    /* TODO: la secuencia compacta de salida */
    return "nop";
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion34_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion34_c.c
/* ia_sesion34_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion34.h. No puede traer main().
 */
#include "api_sesion34.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion34_c.c
/* harness_sesion34_c.c - auditoria automatica de la Sesion 34
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion34_c.c artesano_sesion34_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion34.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 34 - Configuracion de punteros RBP y RSP ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int r = marco_alineado(0, 0); verificar_int("marco_alineado(0, 0)", (long)(0), (long)(r)); }
    { int r = marco_alineado(8, 0); verificar_int("marco_alineado(8, 0)", (long)(16), (long)(r)); }
    { int r = marco_alineado(24, 0); verificar_int("marco_alineado(24, 0)", (long)(32), (long)(r)); }
    { int r = marco_alineado(8, 1); verificar_int("marco_alineado(8, 1)", (long)(8), (long)(r)); }
    { int r = marco_alineado(-1, 0); verificar_int("marco_alineado(-1, 0)", (long)(-1), (long)(r)); }
    { const char *r = epilogo(); verificar_str("epilogo()", "leave; ret", r); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion34.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion34_c.c artesano_sesion34_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion34_c.c ia_sesion34_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion34.md` a `auditorias/auditoria_sesion34.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
